# 1.1 章节介绍：FlashAttention 算子原理

欢迎来到 **FlashAttention 算子开发课程** 的第 1 章！

在大模型推理的世界里，Attention（注意力）是 Transformer 的心脏，也是**最耗时**的算子之一。它有一个与生俱来的痛点：

> 计算量并不离谱，但**访存量离谱**——序列越长，中间结果越庞大，而它们全都躺在最慢的显存（HBM）里写了又读。

FlashAttention（Dao et al., 2022）用一个漂亮的思路解决了这个问题：**把 Attention 整体切块，让中间结果永远留在片上高速缓存，只在最后写出一次结果**。数学上与标准 Attention 完全等价，却快了数倍。

本课程带你理解昇腾 NPU 上 FlashAttention 算子（开源算子仓库 ops-transformer 中的 `FusedInferAttentionScore`）背后的核心思想。而本章，我们先打好地基——**把原理彻底搞懂**。

## 本章学习目标

学完第 1 章，你将能够：

1. **看懂 Attention 在算什么**：用自己的话解释 Q、K、V 的角色，手推「打分 → 归一化 → 加权求和」三步。
2. **理解大模型推理的标配概念**：多头注意力（MHA）、GQA/MQA、KV Cache 分别解决什么问题。
3. **诊断标准实现的性能病根**：画出加速器存储层次，用算术强度（Roofline 模型）判断一个算子是算力瓶颈还是带宽瓶颈，并定量算出 S、P 两个中间矩阵带来的额外访存。
4. **推导 FlashAttention 的两大核心技术**：
   - **Online Softmax**：不看全数据，如何边到边算出精确的 Softmax？
   - **分块计算（Tiling）**：外层 Q 块、内层 KV 块的双层循环如何组织？因果掩码如何做到整块跳过？
5. **动手验证**：用 numpy 实现朴素 / 安全 / 在线三种 Softmax，对拍证明 Online Softmax 与标准 Softmax 逐位一致。

## 前置知识

| 知识点 | 要求 | 说明 |
|--|--|--|
| Python / numpy | 会用 `np.array`、矩阵乘 `@`、`np.exp` | 本章所有实验只用这些 |
| 线性代数 | 知道矩阵乘法和转置 | 不需要更多 |
| 深度学习框架 | **不要求** | 我们从零讲 Attention |
| NPU / CANN | **不要求** | 本章全程可在任意电脑上运行 |

> **零基础读者请放心**：本章假设你没有接触过 Attention，所有概念从「打比方」开始，配有示意图，每一步都有可运行的数值小实验。

若想进一步深入算子的 Ascend C 实现，建议学习 [ascendc_operator_development](../../ascendc_operator_development/README.md) 第 1、2、4 章。


## 环境准备

本章所有数值实验只需 **Python + numpy + Jupyter**，无需 NPU。请先完成以下三步，再开始学习：

### 1. 安装依赖

```bash
python -m pip install numpy ipykernel --user
```

> 已有 Anaconda / miniconda / venv 环境的读者，在对应环境中执行同样命令即可。

### 2. 注册 Jupyter 内核（在 IDE / VS Code / JupyterLab 中运行 notebook 需要）

```bash
python -m ipykernel install --user --name python3 --display-name "Python 3 (fa-course)"
```

注册后在 IDE 中打开任意 `.ipynb`，右上角内核选择 `Python 3 (fa-course)` 即可。

### 3. 验证环境

在 notebook 的任意代码单元中运行：

```python
import numpy as np
print('numpy', np.__version__)   # 输出版本号即说明环境就绪
```

| 依赖 | 用途 | 版本建议 |
|--|--|--|
| numpy | 全部数值实验（矩阵乘、exp、随机数） | ≥ 1.20 |
| ipykernel | 在 IDE / JupyterLab 中交互运行 notebook | ≥ 6.x |

> **离线环境**：可在有网的机器上 `pip download numpy ipykernel -d ./pkgs`，把 `pkgs` 目录拷贝到目标机器后 `pip install --no-index --find-links=./pkgs numpy ipykernel`。

环境就绪后，1.2 ~ 1.5 节的每个代码单元都可以直接运行；1.5 节的实验固定了随机种子，你的输出应与本教程展示的结果一致。

## 章节内容导航

本章共 5 个 Notebook，建议按顺序学习：

| Notebook | 主题 | 核心内容 |
|--|--|--|
| [1.1 章节介绍](01.01_chapter_intro.ipynb)（本页） | 导学 | 目标、前置、路线图 |
| [1.2 Attention 机制基础](01.02_attention_basics.ipynb) | 是什么 | QKV 变换、三步计算、多头、GQA/MQA、KV Cache |
| [1.3 标准 Attention 的访存瓶颈](01.03_attention_bottleneck.ipynb) | 为什么慢 | 存储层次、算术强度、Roofline、三步数据流定量分析 |
| [1.4 FlashAttention 核心原理](01.04_fa_principle.ipynb) | 怎么变快 | Safe Softmax、Online Softmax 三步递推、分块、因果掩码块跳过 |
| [1.5 Online Softmax 数值实验](01.05_online_softmax_experiment.ipynb) | 动手验证 | 三种 Softmax 对拍、数值稳定性、分块误差 |

### 章节主线：一条因果链

```text
1.2 Attention 在算什么（打分→归一化→加权）
        │
        ▼
1.3 标准实现把中间矩阵 S、P 写到 HBM 再读回来 → 带宽被吃光
        │
        ▼
1.4 想让中间结果留在片上，必须解决两个拦路虎：
    ① Softmax 需要整行才能归一化 → Online Softmax 增量修正
    ② 片上放不下整个矩阵 → Q/KV 分块 + 双层循环
        │
        ▼
1.5 用 numpy 亲手验证：切块算出来的结果和整体算的一模一样
```

这条因果链的终点，正是昇腾 NPU 上那个真实的 `FusedInferAttentionScore` 算子 Kernel——它的每一行代码，都是本章某个公式或某张图的工程化落地。

## 学习建议

- **对照图反复推导**：1.4 节的 Online Softmax 三步递推是本章的枢纽，建议推导到能「白板复现」的程度。
- **动手改实验**：1.5 节每个实验都鼓励你改参数（序列长度、分块大小、数值范围），观察结果变化。
- **不必追求一次学完**：概念密集时，先记住「一句话结论」再往下走，回看时再抠细节。

准备好之后，进入 [1.2 Attention 机制基础](01.02_attention_basics.ipynb)——我们从「猫在追什么」的比喻开始。